1. Importing Libraries

In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

from tensorflow import keras
from keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Activation, Dropout, Flatten, Dense
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img
from tensorflow.keras.preprocessing import image_dataset_from_directory

import os
import matplotlib.image as mpimg

2. Importing Dataset

In [ ]:
!pip install opendatasets

In [ ]:
import opendatasets as od
from zipfile import ZipFile
import os

od.download("https://www.kaggle.com/competitions/dogs-vs-cats/data")

# Unzip train.zip if it exists and hasn't been unzipped yet
train_zip_path = 'dogs-vs-cats/train.zip'
train_extracted_path = 'dogs-vs-cats/train'

if os.path.exists(train_zip_path) and not os.path.exists(train_extracted_path):
    print('Extracting train.zip...')
    with ZipFile(train_zip_path, 'r') as zip_ref:
        zip_ref.extractall(os.path.dirname(train_zip_path))
    print('Extracted train.zip')
elif os.path.exists(train_extracted_path):
    print('train directory already exists, skipping extraction.')
else:
    print('train.zip not found or unexpected structure.')

# Add command to inspect the directory structure after extraction
print('\nDirectory structure of dogs-vs-cats:')
!ls -R dogs-vs-cats

3. Visualizating the Data

In [ ]:
import os
import matplotlib.pyplot as plt
from PIL import Image

# Path to train folder
data_dir = "dogs-vs-cats/train" # Corrected path based on typical extraction

# Get all image names
all_images = os.listdir(data_dir)

# Separate dogs and cats
cat_images = [img for img in all_images if img.startswith("cat")]
dog_images = [img for img in all_images if img.startswith("dog")]

# Plot
plt.figure(figsize=(15, 6))

# Show 3 cats
for i in range(3):
    img = Image.open(os.path.join(data_dir, cat_images[i]))
    plt.subplot(2, 3, i + 1)
    plt.imshow(img)
    plt.title("Cat")
    plt.axis("off")

# Show 3 dogs
for i in range(3):
    img = Image.open(os.path.join(data_dir, dog_images[i]))
    plt.subplot(2, 3, i + 4)
    plt.imshow(img)
    plt.title("Dog")
    plt.axis("off")

plt.tight_layout()
plt.show()

4. Splitting Dataset

In [ ]:
import os
import shutil

# Define the new base directory for the structured dataset
base_dir = 'dog-vs-cat-classification'

# Define paths for the new 'cat' and 'dog' directories within the base_dir
cat_dir = os.path.join(base_dir, 'cat')
dog_dir = os.path.join(base_dir, 'dog')

# Create the base directory if it doesn't exist
os.makedirs(base_dir, exist_ok=True)
# Create 'cat' and 'dog' subdirectories if they don't exist
os.makedirs(cat_dir, exist_ok=True)
os.makedirs(dog_dir, exist_ok=True)

# Path to the original extracted images
source_dir = 'dogs-vs-cats/train'

# Move images to their respective class directories
print(f"Moving images from {source_dir} to {base_dir} structure...")
for fname in os.listdir(source_dir):
    if fname.startswith('cat'):
        shutil.move(os.path.join(source_dir, fname), os.path.join(cat_dir, fname))
    elif fname.startswith('dog'):
        shutil.move(os.path.join(source_dir, fname), os.path.join(dog_dir, fname))
print("Image restructuring complete.")

# Now, use image_dataset_from_directory with the newly structured data
train_datagen = image_dataset_from_directory(base_dir,
                                                  image_size=(200,200),
                                                  subset='training',
                                                  seed = 1,
                                                  validation_split=0.1,
                                                  batch_size= 32)
test_datagen = image_dataset_from_directory(base_dir,
                                                  image_size=(200,200),
                                                  subset='validation',
                                                  seed = 1,
                                                  validation_split=0.1,
                                                  batch_size= 32)

5. Model Architecture

In [ ]:
model = tf.keras.models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(200, 200, 3)),
    layers.MaxPooling2D(2, 2),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),

    layers.Flatten(),
    layers.Dense(512, activation='relu'),
    layers.BatchNormalization(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.1),
    layers.BatchNormalization(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.2),
    layers.BatchNormalization(),
    layers.Dense(1, activation='sigmoid')
])

model.summary()

6. Model Compilation and Training

In [ ]:
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

history = model.fit(train_datagen,
          epochs=10,
          validation_data=test_datagen)

Epoch 1/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 1114s 2s/step - accuracy: 0.5639 - loss: 0.7677 - val_accuracy: 0.5208 - val_loss: 0.7266
Epoch 2/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 1169s 2s/step - accuracy: 0.6159 - loss: 0.6620 - val_accuracy: 0.5076 - val_loss: 0.7890
Epoch 3/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 1166s 2s/step - accuracy: 0.6666 - loss: 0.6184 - val_accuracy: 0.6700 - val_loss: 0.6160
Epoch 4/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 1133s 2s/step - accuracy: 0.6846 - loss: 0.5920 - val_accuracy: 0.5220 - val_loss: 1.2915
Epoch 5/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 1154s 2s/step - accuracy: 0.6584 - loss: 0.6194 - val_accuracy: 0.5672 - val_loss: 0.8426
Epoch 6/10
136/704 ━━━━━━━━━━━━━━━━━━━━ 14:26 2s/step - accuracy: 0.6924 - loss: 0.5815

7. Model Evaluation

In [ ]:
history_df = pd.DataFrame(history.history)
history_df.loc[:, ['loss', 'val_loss']].plot()
history_df.loc[:, ['accuracy', 'val_accuracy']].plot()
plt.show()

8. Model Testing and Prediction

In [ ]:
def predict_image(image_path):
    img = image.load_img(image_path, target_size=(200, 200))
    plt.imshow(img)
    img = image.img_to_array(img)
    img = np.expand_dims(img, axis=0)

    result = model.predict(img)
    print("Dog" if result >= 0.5 else "Cat")

predict_image('dog-vs-cat-classification/cat/cat.320.jpg')
predict_image('dog-vs-cat-classification/dog/dog.5510.jpg')